# Lab 7 — Large Language Models (LLMs)



## The Three Main Transformer Model Families

| Family | Example Models | Main Use |
|---|---|---|
| **Encoder-only** | BERT, RoBERTa | Classification, sentiment analysis, NER |
| **Encoder–decoder** | T5, BART | Translation, summarization, question answering |
| **Decoder-only** | GPT family | Text generation, conversation, completion |


## 1. Setup


In [1]:
%pip install -q transformers sentencepiece torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import pipeline, set_seed

print("Libraries loaded successfully!")


Libraries loaded successfully!


# Part A — Decoder-Only Model

## Decoder-only example: GPT-style Text Generation

A decoder-only model predicts the next token based on previous tokens.

We will use **DistilGPT-2**, a small GPT-style model.


In [3]:
generator = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"

output = generator(prompt, max_new_tokens=30, do_sample=False)

print(output[0]["generated_text"])

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Natural language processing is a common feature of the language processing process.























# Part B — Encoder–Decoder Model

## Encoder–decoder example: Instruction / Text-to-Text Generation

Encoder–decoder models contain two main parts:

1. The **encoder** reads the input.
2. The **decoder** produces the output.

We will use **FLAN-T5-small**.


In [4]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

prompt = "Translate to French: I love natural language processing."

# Convert text into tokens
output = model.generate(**tokenizer(prompt, return_tensors="pt"), max_new_tokens=40)

# Convert tokens back to text
print(tokenizer.decode(output[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Je s'agissant de la technologie de linguage naturelle.


# Part C — Encoder-Only Model

## Encoder-only example: Sentiment Classification

Encoder-only models read the full input and create a representation of it.

They are very useful when the task is to **understand or classify text**.

Here we use a DistilBERT sentiment model.


In [5]:
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

text = "This NLP lab is easy and interesting."

result = classifier(text)
print(result)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.999778687953949}]


# Tasks

Complete the following tasks.

## Task 1 — Model Families

For each task below, choose the most suitable model family:

1. Sentiment classification
2. Machine translation
3. Open-ended text generation
4. Named Entity Recognition
5. Text summarization

In [6]:
# Task 1 - which family fits which task
answers = {
    "1. Sentiment classification":   "Encoder-only      (BERT / RoBERTa)",
    "2. Machine translation":        "Encoder-decoder   (T5 / BART)",
    "3. Open-ended text generation": "Decoder-only      (GPT family)",
    "4. Named Entity Recognition":   "Encoder-only      (BERT / RoBERTa)",
    "5. Text summarization":         "Encoder-decoder   (T5 / BART)",
}

for task, family in answers.items():
    print(f"{task:<32} -> {family}")

print()
print("Reasoning:")
print("  Encoder-only  - the task is to understand the whole input and return a")
print("                  label or a tag per token. Bidirectional attention lets each")
print("                  token see the full sentence, which is what classification")
print("                  and NER need.")
print("  Enc-decoder   - input and output are both sequences and they are different")
print("                  texts. The encoder reads the source once, the decoder writes")
print("                  the target while attending back to it.")
print("  Decoder-only  - there is no separate input to encode; the output *is* the")
print("                  continuation of the input, so one causal stack is enough.")

1. Sentiment classification      -> Encoder-only      (BERT / RoBERTa)
2. Machine translation           -> Encoder-decoder   (T5 / BART)
3. Open-ended text generation    -> Decoder-only      (GPT family)
4. Named Entity Recognition      -> Encoder-only      (BERT / RoBERTa)
5. Text summarization            -> Encoder-decoder   (T5 / BART)

Reasoning:
  Encoder-only  - the task is to understand the whole input and return a
                  label or a tag per token. Bidirectional attention lets each
                  token see the full sentence, which is what classification
                  and NER need.
  Enc-decoder   - input and output are both sequences and they are different
                  texts. The encoder reads the source once, the decoder writes
                  the target while attending back to it.
  Decoder-only  - there is no separate input to encode; the output *is* the
                  continuation of the input, so one causal stack is enough.


## Task 2 — Encoder-only Model

Use the sentiment classifier on these three sentences:

```text
The movie was excellent.
The lecture was difficult to understand.
I enjoyed learning about transformers.
```

Print the predicted label and score for each sentence.

In [7]:
# Task 2 - the encoder-only sentiment classifier on three sentences
sentences = [
    "The movie was excellent.",
    "The lecture was difficult to understand.",
    "I enjoyed learning about transformers.",
]

for sentence in sentences:
    result = classifier(sentence)[0]
    print(f"{sentence:<45} {result['label']:<9} score={result['score']:.4f}")

print()
print("Reading the second sentence: the model is just as confident (0.9997) as")
print("it is on the other two. It is keying on the polarity of 'difficult' and")
print("not on what the sentence is *about*, which is why a negative opinion of a")
print("lecture is scored the same way as a negative product review.")
print()
print("Two limits worth noting:")
print("  - this checkpoint was fine-tuned on SST-2, which has only two labels, so")
print("    every sentence is forced to POSITIVE or NEGATIVE and a neutral or mixed")
print("    sentence cannot be represented at all;")
print("  - the score is a confidence, not a calibrated probability. 0.9997 does")
print("    not mean the model is right 99.97% of the time on this kind of text.")

The movie was excellent.                      POSITIVE  score=0.9999
The lecture was difficult to understand.      NEGATIVE  score=0.9997
I enjoyed learning about transformers.        POSITIVE  score=0.9997

Reading the second sentence: the model is just as confident (0.9997) as
it is on the other two. It is keying on the polarity of 'difficult' and
not on what the sentence is *about*, which is why a negative opinion of a
lecture is scored the same way as a negative product review.

Two limits worth noting:
  - this checkpoint was fine-tuned on SST-2, which has only two labels, so
    every sentence is forced to POSITIVE or NEGATIVE and a neutral or mixed
    sentence cannot be represented at all;
  - the score is a confidence, not a calibrated probability. 0.9997 does
    not mean the model is right 99.97% of the time on this kind of text.


## Task 3 — Encoder–Decoder Model

Use the FLAN-T5 model to perform the following:

- Summarize a short paragraph.


In [8]:
# Task 3 - encoder-decoder (FLAN-T5) summarization
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()      # keep the load output readable

paragraph = (
    "Natural language processing is a branch of computer science and artificial "
    "intelligence that studies how computers can understand and generate human "
    "language. Early systems relied on hand-written grammar rules, which were "
    "brittle and expensive to maintain. From the 1990s onward the field moved to "
    "statistical methods trained on large text corpora, and more recently to deep "
    "neural networks and transformer models. These models are pretrained on huge "
    "amounts of unlabelled text and then fine-tuned for specific tasks such as "
    "translation, summarization, question answering and sentiment analysis. The "
    "same pretrained model can often be reused for many tasks, which has made NLP "
    "far more accessible than it was when every task needed its own purpose-built "
    "system."
)

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small")

inputs = tokenizer("summarize: " + paragraph, return_tensors="pt", truncation=True)

summary = model.generate(**inputs, max_new_tokens=60, num_beams=4, early_stopping=True)
print(tokenizer.decode(summary[0], skip_special_tokens=True))
print()
print("What this shows: flan-t5-small followed the instruction - it produced a")
print("single-sentence summary rather than continuing the text, which is the")
print("point of using an encoder-decoder here. It is also a small model, and the")
print("summary keeps only the opening claim and drops the whole 1990s-to-")
print("transformers arc, which is the part that actually matters. Reading the")
print("input and writing a faithful short version of it are different problems,")
print("and 80M parameters is not enough for the second one.")

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In the 1990s, natural language processing was a branch of computer science and artificial intelligence that studies how computers can understand and generate human language.

What this shows: flan-t5-small followed the instruction - it produced a
single-sentence summary rather than continuing the text, which is the
point of using an encoder-decoder here. It is also a small model, and the
summary keeps only the opening claim and drops the whole 1990s-to-
transformers arc, which is the part that actually matters. Reading the
input and writing a faithful short version of it are different problems,
and 80M parameters is not enough for the second one.



## Task 5 — Short Reflection

Answer in 3–5 sentences:

**Why is a decoder-only model suitable for a chatbot such as ChatGPT?**


A chatbot never produces a whole answer at once, it emits it one token at a time, and each token should depend only on what has already been said. That is exactly the causal mask of a decoder-only model, so generation and the model's only training objective - predict the next token - are the same operation. The conversation, including the user's turn, is just a token sequence in the same context window, so there is no separate encoder to hand an input to and no encoder-decoder boundary to design around. One architecture trained on raw text therefore covers open-ended chat, summarization, and classification, which is why the GPT family scaled into assistants rather than task-specific pipelines. Encoder-only models cannot generate at all, and encoder-decoder models would need the reply re-encoded at every step.

# Optional Challenge

Try generating text several times with the same prompt.

Remove:

```python
set_seed(42)
```

and run the sampling cell multiple times.

### Question

Why do you sometimes get different outputs even when the prompt is the same?

**Hint:** Sampling introduces randomness when choosing the next token.


## Optional Challenge - Sampling

In [9]:
# Optional challenge - sampling
from transformers import set_seed, pipeline

set_seed(42)                                   # comment this line out and re-run
sampler = pipeline("text-generation", model="distilgpt2")

prompt = "Natural language processing is"
for run in range(3):
    out = sampler(prompt, max_new_tokens=30, do_sample=True, temperature=0.9, top_k=50)
    print(f"run {run+1}: {out[0]['generated_text']}")
    print()

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_k', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


run 1: Natural language processing is more prevalent worldwide than is used in traditional languages, because it is harder to understand and understand these languages. The research team found that most languages are languages

run 2: Natural language processing is an area of interest for both of us and the other species.



run 3: Natural language processing is a fundamental part of the science and culture of the United States. As a result, our nation› has witnessed a profound increase in education, economic



**Optional challenge - why the outputs differ**

With `do_sample=True` the next token is *drawn* from the probability distribution over the vocabulary instead of taking the highest-probability token. `temperature` reshapes that distribution (higher temperature flattens it, so unlikely tokens get picked more often) and `top_k`/`top_p` cut off the tail before sampling. So the same prompt walks a different random path each run. `set_seed(42)` pins PyTorch's random number generator to a fixed starting state, which makes those draws reproducible, and removing it lets the run vary again. Greedy decoding (`do_sample=False`, what Part A uses) has no randomness at all: it always takes the argmax, so repeated runs are identical.